# このノートブックについて

このノートブックでは前回のノートブックで実装及び学習による性能評価を完了したCareFLを用いて介入分布からの生成と反事実推論を行っていくことを目的としたノートブックである。

# 1. CareFLを用いた介入分布からの生成とその評価

In [20]:
import os

os.chdir('/home/hashikami/projects/Diff')

## 1.1 介入分布からの生成実装

In [44]:
import importlib 
from carefl.models import carefl

importlib.reload(carefl)

from carefl.models.carefl import CAREFL
import yaml
import argparse

with open('config/config.yaml', 'r') as f:
    config_raw = yaml.load(f, Loader=yaml.FullLoader)
    
def dict2namespace(config):
    namespace = argparse.Namespace()
    for key, value in config.items():
        if isinstance(value, dict):
            new_value = dict2namespace(value)
        else:
            new_value = value
        setattr(namespace, key, new_value)
    return namespace

config = dict2namespace(config_raw)
carefl = CAREFL(config)

In [45]:
flow, loss_vals = carefl._train()

  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [00:25<00:00,  7.89it/s]


In [46]:
carefl.B_true

array([[0., 1., 1.],
       [0., 0., 0.],
       [0., 1., 0.]])

In [24]:
def predict_intervention(carefl, int_idx, int_val, n_samples=100):
    
    if isinstance(int_idx, (int, np.integer)):
        int_idx = [int_idx]
    if np.isscalar(int_val):
        int_val = [float(int_val)]
    assert len(int_idx) == len(int_val)

    int_idx = list(int_idx)
    int_val = list(int_val)
    device = carefl.device
    
    flows = carefl.flow.flow.flows
    z = carefl.flow.prior.sample((n_samples,)).to(device)
    for affine in flows[::-1]:
        trans_idx = affine.trans_idx[0]
        if trans_idx in int_idx:
            z[:, trans_idx] = int_val[int_idx.index(trans_idx)]
        else:
            z, _ = affine.backward(z)
            
    return z

In [42]:
import numpy as np

int_idx = 1
int_val = 3.0
x = predict_intervention(carefl, int_idx, int_val, n_samples=1000)

In [43]:
x.mean(0)

tensor([0.0894, 3.0000, 3.6989], device='cuda:0', grad_fn=<MeanBackward1>)

In [36]:
xs = carefl.flow.sample(1000)
x = xs[-1]
x.mean(0)

tensor([ 0.0216, -0.1451,  0.1216], device='cuda:0', grad_fn=<MeanBackward1>)

## 1.2 モジュールの動作確認

In [51]:
int_idx = 2
int_val = 3.0

x = carefl.predict_intervention(int_idx, int_val, n_samples=1000)

In [52]:
x.mean(0)

tensor([-0.0366, -3.1375,  3.0000], device='cuda:0', grad_fn=<MeanBackward1>)